 ### 1. DATA UNDERSTANDING

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import sqlite3
from scipy import stats
from os.path import exists

In [2]:
conn = sqlite3.connect('../data/im/im.db')
pd.read_sql('''
SELECT name FROM sqlite_master
    WHERE type = 'table'
''',conn)

,name


In [3]:
movie_basics_df = pd.read_sql("""
    SELECT * FROM movie_basics
""",conn)
movie_basics_df.head()

DatabaseError: Execution failed on sql '
    SELECT * FROM movie_basics
': no such table: movie_basics

In [ ]:
movie_ratings_df = pd.read_sql("""
       SELECT * FROM movie_ratings
""",conn)
movie_ratings_df

In [ ]:
df_tn_budgets = pd.read_csv("../data/tn.movie_budgets.csv.gz", compression='gzip')
df_rt_reviews = pd.read_csv("../data/rt.reviews.tsv.gz", compression='gzip',sep='\t',low_memory=False,encoding='latin1')
df_rt_info = pd.read_csv("../data/rt.movie_info.tsv.gz", compression='gzip',sep='\t', low_memory=False)
df_bom_gross = pd.read_csv("../data/bom.movie_gross.csv.gz", compression='gzip')
df_tmdb_movies = pd.read_csv("../data/tmdb.movies.csv.gz", compression='gzip')

In [ ]:
df_rt_info.columns

#### 1.1 Data Understanding on tn.movie_budgets

In [ ]:
print("\n--- 2. Initial Data Understanding (Raw Data) ---")

print("\n--- df_tn_budgets (Movie Budgets) ---")
df_tn_budgets.head()


In [ ]:
df_tn_budgets.info()

In [ ]:
print("Checking Duplicates for Budget")
df_tn_budgets.duplicated().sum()

In [ ]:
print("-------------Missing values\n", df_tn_budgets.isnull().sum())

#### 1.2 Data Understanding for movies_info

In [ ]:
print("\n--- df_rt_info (Rotten Tomatoes Movie Info) ---")
df_rt_info.head(10)

In [ ]:
 df_rt_info.info()

In [ ]:
print("Checking duplicates for movies_info")
df_rt_info.duplicated().sum()

In [ ]:
print("Missing values:\n", df_rt_info.isnull().sum())

#### 1.3 Data Understanding for Bom_gross

In [ ]:
print("\n--- df_bom_gross (Box Office Mojo Gross) ---")
df_bom_gross.head()

In [ ]:
df_bom_gross.info()

In [ ]:
print("Checking duplicates for bom_gross")
df_bom_gross.duplicated().sum()

In [ ]:
print("Missing values:\n", df_bom_gross.isnull().sum())

#### 1.4 Data Understanding for Reviews

In [ ]:
print("\n--- df_rt_reviews (Rotten Tomatoes Reviews) ---")
df_rt_reviews.head(20)

In [ ]:
df_rt_reviews.info()

In [ ]:
print("Checking duplicates for reviews")
df_rt_reviews.duplicated().sum()

In [ ]:
print("Missing values:\n", df_rt_reviews.isnull().sum())

#### 1.5 Data Understanding for tmdb

In [ ]:
print("\n--- df_tmdb_movies (TMDB Movies) ---")
df_tmdb_movies.tail()

In [ ]:
df_tmdb_movies.info()

In [ ]:
print("Checking duplicates for tmdb")
df_tmdb_movies.duplicated().sum()

In [ ]:
print("Missing values:\n", df_tmdb_movies.isnull().sum())

### 2.DATA CLEANING

#### Basically Data Type Conversions and Missing Value Handling

### 2.1 tn_movie_budget 
***Convert currency strings to numeric, and also parsing dates***

In [ ]:
# Clean financial columns in df_tn_budgets
for col in ['production_budget', 'domestic_gross', 'worldwide_gross']:
    if col in df_tn_budgets.columns and df_tn_budgets[col].dtype == 'object':
        df_tn_budgets[col] = df_tn_budgets[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
        df_tn_budgets[col] = pd.to_numeric(df_tn_budgets[col], errors='coerce')

# Verify the data types after cleaning
print(df_tn_budgets.info())

In [ ]:
df_tn_budgets['release_date'] = pd.to_datetime(df_tn_budgets['release_date'], errors='coerce')

In [ ]:
# Extract release year from tn
df_tn_budgets['release_year'] = df_tn_budgets['release_date'].dt.year

In [ ]:
df_tn_budgets = df_tn_budgets[df_tn_budgets['release_year'] > 2010]

In [ ]:
df_tn_budgets.release_year.value_counts()

In [ ]:
print("df_tn_budgets after cleaning:")
print()
df_tn_budgets.info()

In [ ]:
df_tn_budgets.to_csv('cleaned_df_tn_budgets')

### 2.2 bom_gross
***Convert currency strings to numeric, fill missing 'foreign_gross' with 0.***


***fill missing 'studio' with 'Uknown.***


***drop rows with missing 'domestic_gross***

In [ ]:
print("\nCleaning df_bom_gross...")
for col in ['domestic_gross', 'foreign_gross']:
        if col in df_bom_gross.columns and df_bom_gross[col].dtype == 'object':
            df_bom_gross[col] = df_bom_gross[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
            df_bom_gross[col] = pd.to_numeric(df_bom_gross[col], errors='coerce')

In [ ]:
df_bom_gross.drop(columns='foreign_gross', inplace=True)

In [ ]:
if 'studio' in df_bom_gross.columns:
        initial_missing_studio = df_bom_gross['studio'].isnull().sum()
        df_bom_gross['studio'] = df_bom_gross['studio'].fillna('Unknown')
        print(f"Filled {initial_missing_studio} missing 'studio' values with 'Unknown'.")


In [ ]:
if 'domestic_gross' in df_bom_gross.columns:
        initial_rows_bom_domestic = len(df_bom_gross)
        df_bom_gross.dropna(subset=['domestic_gross'], inplace=True)
        print(f"df_bom_gross: Dropped {initial_rows_bom_domestic - len(df_bom_gross)} rows due to missing 'domestic_gross'.")

In [ ]:
print("df_bom_gross after cleaning:")
print()
df_bom_gross.info()

In [ ]:
print("Missing values after cleaning:\n", df_bom_gross.isnull().sum())

In [ ]:
df_bom_gross.to_csv('cleaned_df_bom_gross')

### 2.3 movies_info(rt_info)
***Convert 'box_office' to numeric, parse 'theater_date'***

***Convert 'runtime' to numeric and then fill missing***

In [ ]:
print("\nCleaning df_rt_info...")
if 'box_office' in df_rt_info.columns and df_rt_info['box_office'].dtype == 'object':
        df_rt_info['box_office'] = df_rt_info['box_office'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
        df_rt_info['box_office'] = pd.to_numeric(df_rt_info['box_office'], errors='coerce')
if 'theater_date' in df_rt_info.columns:
        df_rt_info['theater_date'] = pd.to_datetime(df_rt_info['theater_date'], errors='coerce')

In [ ]:
for col in ['synopsis', 'rating', 'genre', 'director', 'writer', 'studio']:
        if col in df_rt_info.columns:
            initial_missing_count = df_rt_info[col].isnull().sum()
            if initial_missing_count > 0:
                df_rt_info[col] = df_rt_info[col].fillna('Unknown')
                print(f"Filled {initial_missing_count} missing '{col}' values with 'Unknown'.")


In [ ]:
if 'runtime' in df_rt_info.columns:
        # First, remove " minutes" string if present
        if df_rt_info['runtime'].dtype == 'object':
            df_rt_info['runtime'] = df_rt_info['runtime'].astype(str).str.replace(' minutes', '', regex=False)
        # Convert to numeric, coercing errors (e.g., non-numeric strings) to NaN
        df_rt_info['runtime'] = pd.to_numeric(df_rt_info['runtime'], errors='coerce')
        initial_missing_runtime = df_rt_info['runtime'].isnull().sum()
        if initial_missing_runtime > 0:
            median_runtime = df_rt_info['runtime'].median()
            df_rt_info['runtime'] = df_rt_info['runtime'].fillna(median_runtime)
            print(f"Filled {initial_missing_runtime} missing 'runtime' values with median ({median_runtime}).")


In [ ]:
print("df_rt_info after cleaning:")
df_rt_info.info()


In [ ]:
print("Missing values after cleaning:\n", df_rt_info.isnull().sum())

In [ ]:
df_rt_info.rating.value_counts()

In [ ]:
df_rt_info = df_rt_info.drop(["theater_date", "dvd_date","currency","box_office"], axis=1)

In [ ]:
df_rt_info.to_csv('cleaned_df_rt_info')

***For the theatre and dvd:These are extra date columns that aren't critical. We already have complete and reliable release date info from release_date_tn (in tn.movie_budgets)***

***For the currency and box office:These are secondary financial details from Rotten Tomatoes. We don’t rely on them for final financial analysis. Instead, we use more complete and prioritized data from tn.movie_budgets and bom.movie_gross for Worldwide_Gross***

### 2.4 Cleaning rt_reviews
***Parsing date.***

***Fill missing 'review' text with a placeholder***

***Fill missing 'rating' with median***

***Fill missing 'critic' and 'publisher' with 'Unknown'***

In [ ]:
print("\nCleaning df_rt_reviews...")
if 'date' in df_rt_reviews.columns:
        df_rt_reviews['date'] = pd.to_datetime(df_rt_reviews['date'], errors='coerce')

In [ ]:
if 'review' in df_rt_reviews.columns:
        initial_missing_review = df_rt_reviews['review'].isnull().sum()
        if initial_missing_review > 0:
            df_rt_reviews['review'] = df_rt_reviews['review'].fillna('No Review Text')
            print(f"Filled {initial_missing_review} missing 'review' values with 'No Review Text'.")

In [ ]:
if all(col in df_rt_reviews.columns for col in ['id', 'review', 'critic','publisher', 'date']):
        initial_rows_rt_reviews = len(df_rt_reviews)
        df_rt_reviews.drop_duplicates(subset=['id', 'review', 'critic','publisher', 'date'])
        print(f"df_rt_reviews: Removed {initial_rows_rt_reviews - len(df_rt_reviews)} duplicates based on 'id', 'review','publisher', 'critic', and 'date'.")
else:
        print("df_rt_reviews: Skipping duplicate check due to missing 'id', 'review', 'critic', or 'date' columns.")

In [ ]:
df_rt_reviews.drop(columns='rating', inplace=True)

In [ ]:
df_rt_reviews.fresh.value_counts()

In [ ]:
for col in ['critic', 'publisher']:
        if col in df_rt_reviews.columns:
            initial_missing_count = df_rt_reviews[col].isnull().sum()
            if initial_missing_count > 0:
                df_rt_reviews[col] = df_rt_reviews[col].fillna('Unknown ' + col.capitalize())
                print(f"Filled {initial_missing_count} missing '{col}' values with 'Unknown {col.capitalize()}'.")


In [ ]:
print("df_rt_reviews after cleaning:")
print()
print(df_rt_reviews.info())
print()
print(f"Duplicates:", df_rt_reviews.duplicated().sum())

In [ ]:
print("Missing values after cleaning:\n", df_rt_reviews.isnull().sum())

In [ ]:
df_rt_reviews.to_csv("cleaned_df_rt_reviews")

### 2.5 tmbd_movies Cleaning
***Parse 'release_date' column***

In [ ]:
df_tmdb_movies['release_date'] = pd.to_datetime(df_tmdb_movies['release_date'], errors='coerce')
df_tmdb_movies['release_year'] = df_tmdb_movies['release_date'].dt.year

In [ ]:
print("df_tmdb_movies after cleaning:")
print()
df_tmdb_movies.info()

In [ ]:
print("Missing values after cleaning:\n", df_tmdb_movies.isnull().sum())

In [ ]:
df_tmdb_movies.to_csv("cleaned_df_tmdb_movies")

### Merging and cleaning the database data

In [ ]:
movie_with_ratings = pd.merge(movie_ratings_df,movie_basics_df,left_on='movie_id',right_on='movie_id')
movie_with_ratings.head()

In [ ]:
movie_with_ratings.isnull().sum()

In [ ]:
movie_with_ratings = movie_with_ratings.dropna()

In [ ]:
movie_with_ratings.info()

In [ ]:
movie_with_ratings.to_csv("cleaned_merged_movie_wih_ratings")

### 3. Feature Engineering
*** Calculate Review_Count from df_rt_reviews***

In [ ]:
# Compute Review_Count and Freshness_Score
review_counts = df_rt_reviews.groupby('id').agg(
    Review_Count=('fresh', 'count'),
    Freshness_Score=('fresh', lambda x: (x == 'fresh').sum() / len(x))
).reset_index()

In [ ]:
review_counts

In [ ]:
review_counts.columns = review_counts.columns.str.lower()

***Merging Reviews and Info***

In [ ]:
df_rt_combined = pd.merge(df_rt_info, review_counts, on='id', how='left')
df_rt_combined

In [ ]:
df_rt_combined.isnull().sum()

In [ ]:
# Fill missing review_count with 0
df_rt_combined['review_count'] = df_rt_combined['review_count'].fillna(0)

# Fill missing freshness_score with median
median_fresh = df_rt_combined['freshness_score'].median()
df_rt_combined['freshness_score'] = df_rt_combined['freshness_score'].fillna(median_fresh)

In [ ]:
# Standardize column names to lowercase
df_rt_combined.columns = df_rt_combined.columns.str.lower()

In [ ]:
df_rt_combined.to_csv("cleaned_merged_reviewinfo")

***Merging tmdb and tn_budgets***

In [ ]:
#for differentiation I renamed the 'title' to 'tmdb_title' to avoid problems when merging
df_tmdb_movies.rename(columns={'title': 'tmdb_title'}, inplace=True)

In [ ]:
#Renamed movie in tn_budget for easy understanding when merging with tmdb
df_tn_budgets.rename(columns={'movie': 'title'}, inplace=True)

In [ ]:
# Standardize title fields in both datasets
df_tn_budgets['title_clean'] = df_tn_budgets['title'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
df_tmdb_movies['tmdb_title_clean'] = df_tmdb_movies['tmdb_title'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
movie_with_ratings['primary_title_clean'] = movie_with_ratings['primary_title'].str.lower().str.strip().str.replace(r'[^\w\s]', '', regex=True)
# # Merge on the cleaned titles
merged = pd.merge(df_tn_budgets, df_tmdb_movies, left_on=['title_clean'], right_on=['tmdb_title_clean'], how='left')

In [ ]:
# Financial and temporal features
merged['Profit'] = merged['worldwide_gross'] - merged['production_budget']
merged['ROI'] = merged['Profit'] / merged['production_budget']
merged['Domestic_vs_Worldwide_Ratio'] = merged['domestic_gross'] / merged['worldwide_gross']
merged['Release_Month'] = merged['release_date_x'].dt.month
merged['Day_of_Week'] = merged['release_date_x'].dt.day_name()
merged['foreign_gross'] = merged['worldwide_gross'] - merged['domestic_gross']

In [ ]:
merged.columns

In [ ]:
# Drop suffix columns and low-value metadata
low_value_cols = [
    'id_y', 'tmdb_title', 'original_title', 'original_language',
    'release_date_y', 'release_date_x','foreign_gross', 'tmdb_title_clean', 'release_year_y',
    'title'
   ,'Unnamed: 0'
]
merged.drop(columns=low_value_cols, errors='ignore', inplace=True)
merged.head()

In [ ]:
merged.isnull().sum()

In [ ]:
merged.columns=merged.columns.str.lower()

***The column Domestic_vs_Worldwide_Ratio has 370 missing values, which means in those rows, either worldwide_gross, domestic_gross, or both were missing or zero .....and dividing by zero isn’t really wise basically.So decided to drop bcz it'll help in actually rolling out data that may be irrelevant in our analysis...basically I mean like having domestic = 923,456,324 and worldwide = 0...so this zii**


In [ ]:
merged = merged[merged['domestic_vs_worldwide_ratio'].notnull()]

In [ ]:
merged.isnull().sum()

In [ ]:
merged.to_csv('merged_tn_tmdb')

### Director and Writer Metrics

In [ ]:
review_metrics = df_rt_reviews.groupby('id').agg(
    review_count=('fresh', 'count'),
    freshness_score=('fresh', lambda x: (x == 'fresh').sum() / len(x))
).reset_index()

In [ ]:
# Merge review metrics with director and writer
review_metrics_full = pd.merge(df_rt_info[['id', 'director', 'writer']], review_metrics, on='id', how='left')

In [ ]:
# Group by director
director_metrics = review_metrics_full.groupby('director').agg(
    director_avg_freshness=('freshness_score', 'mean'),
    director_avg_reviews=('review_count', 'mean'),
    director_movie_count=('id', 'count')
).reset_index()

# Group by writer
writer_metrics = review_metrics_full.groupby('writer').agg(
    writer_avg_freshness=('freshness_score', 'mean'),
    writer_avg_reviews=('review_count', 'mean'),
    writer_movie_count=('id', 'count')
).reset_index()

In [ ]:
director_metrics

In [ ]:
director_metrics.isnull().sum()

In [ ]:
top_directors_by_reviews = director_metrics.sort_values(by='director_avg_reviews', ascending=False).head(10)
top_directors_by_reviews

In [ ]:
top_writers_by_reviews = writer_metrics.sort_values(by='writer_avg_reviews', ascending=False).head(10)
top_writers_by_reviews

In [ ]:
merged

In [ ]:
merged2 =merged.merge(movie_with_ratings, left_on='title_clean', right_on='primary_title_clean', how='inner')

In [ ]:
#sort dataframe by movies alphabetically
merged2 = merged2.sort_values(by='title_clean', ascending=True)
merged2.head()

In [ ]:
merged2 = merged2.groupby('title_clean')
merged2 = merged2.first()

In [ ]:
merged2

In [ ]:
#for analysis purposes, change all multiple genre observations into the string "mix"
merged2.loc[merged2['genres'].str.contains(','), 'genres'] = 'mix'
merged2.head()

In [ ]:
merged2.to_csv('merged_with_db')

## 2. EXPLORATORY DATA ANALYSIS
 In this section, we will perform Univariate, Bivariate, and Multivariate Analysis with appropriate visualizations for each.

### 2.1 Univariate Analysis

### Distribution of Production Budgets
This histogram reveals the range of production investments. Most movies have budgets clustered at the lower end, with a long tail of big-budget productions.

In [ ]:
# Distribution of production budgets
import seaborn as sns
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
sns.histplot(merged2['production_budget'], bins=30, kde=True)
plt.title('Distribution of Production Budgets')
plt.xlabel('Production Budget')
plt.ylabel('Frequency')
plt.show()

### ROI Distribution After Removing Outliers
This visualization gives a cleaner view of Return on Investment (ROI), emphasizing typical performance without skew from extreme hits or flops.

In [ ]:
# Step 1: Calculate IQR for ROI
Q1 = merged2['roi'].quantile(0.25)
Q3 = merged2['roi'].quantile(0.75)
IQR = Q3 - Q1

# Step 2: Define upper and lower bounds for outlier removal
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Step 3: Filter the DataFrame to remove outliers
df_filtered = merged2[(merged2['roi'] >= lower_bound) & (merged2['roi'] <= upper_bound)]

# Step 4: Plot the histogram without outliers
plt.figure(figsize=(10, 5))
sns.histplot(df_filtered['roi'], bins=50, color='skyblue', kde=True)
plt.title('Distribution of ROI (Outliers Removed)', fontsize=14)
plt.xlabel('ROI', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.grid(True, linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

### Distribution of Review Count
Review count helps estimate public or critical interest. The majority of films receive relatively few reviews, with a few standouts getting extensive attention.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(review_counts['review_count'], bins=30, kde=True)
plt.title('Distribution of Review Count')
plt.xlabel('Review Count')
plt.ylabel('Frequency')
plt.show()

### Distribution of Freshness Score
Freshness scores show critical reception. Most movies cluster around average scores, with fewer achieving critical acclaim.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(review_counts['freshness_score'], bins=30, kde=True)
plt.title('Distribution of Freshness Score')
plt.xlabel('Freshness Score')
plt.ylabel('Frequency')
plt.show()

### Bivariate Analysis

### ROI vs. Profit

In [ ]:
plt.figure(figsize=(10,5))
sns.scatterplot(x=merged2['roi'], y=merged2['profit'])
plt.title('ROI vs Profit')
plt.xlabel('ROI')
plt.ylabel('Profit')
plt.show()

This scatter plot explores the relationship between ROI and Profit. Although correlated, it's clear that even high-ROI movies can yield modest profits when budgets are small.

### ROI vs. Budget

In [ ]:
import numpy as np
fig, ax = plt.subplots()
fig.set_size_inches(20,10)
x = merged2["production_budget"]
y = merged2["roi"]
df = pd.DataFrame({'x': merged2["production_budget"], 'y': merged2["roi"]})
plt.xlabel('Budget (In Millions Of U.S. Dollars)', fontsize = 20 , weight ='bold')
plt.ylabel('Return On Investment (In Billions Of U.S. Dollars) ', fontsize = 20, weight ='bold')
# plt.title('ROI Vs Budget', fontsize = 17 , weight='bold')
plt.xticks(fontsize=14, rotation=0)
plt.yticks(fontsize=14, rotation=0)
ax.scatter(df.x, df.y, c=np.sign(df.y), cmap="coolwarm")
plt.show()

This scatter plot shows how Return on Investment varies with production budget. Smaller budgets often achieve higher ROI, while large budgets show more spread.

In [ ]:
month_av = merged2[['release_month', 'worldwide_gross']]
month_av_grp = month_av.groupby('release_month', as_index=False).mean()
month_av_grp

### Monthly Gross Trends

In [ ]:
import matplotlib.ticker as ticker
fig, ax = plt.subplots()
fig.set_size_inches(20,10)

ax = sns.lineplot(x="release_month", y="worldwide_gross", marker="o", ci=0, markersize=15, data=month_av_grp)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter(1))
plt.yticks(ax.get_yticks(), ax.get_yticks()/1000000)
plt.ylabel('Average Movie Gross Revenue (In Millions Of U.S. Dollars)', fontsize=15)
plt.xlabel('Movie Release Month', fontsize=15)
sns.set(style='whitegrid', font_scale=1.2)
plt.show()


This line plot tracks average worldwide gross by release month, revealing clear seasonal peaks (e.g., summer and holiday releases).

### Vote Count vs. Worldwide Gross

In [ ]:
#scatterplot of vote_count vs. worldwide_gross
fig, ax = plt.subplots()
fig.set_size_inches(18,12)

ax = sns.scatterplot(x="vote_count", y="worldwide_gross", marker="o", ci=68, data=merged2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(2000))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set(xlabel='Number Of Votes', ylabel='Movie Gross Revenue (In Hundreds Of Millions Of U.S. Dollars)')
sns.set(style='whitegrid', font_scale=1.5)
plt.show()

This scatter plot evaluates if audience voting activity correlates with box office revenue. Movies with more votes tend to have higher gross revenue.

### Average Rating vs. Worldwide Gross

In [ ]:
#scatterplot of vote_average vs. worldwide_gross
fig, ax = plt.subplots()
fig.set_size_inches(60,12)

ax = sns.barplot(x="vote_average", y="worldwide_gross", ci=None, data=merged)
ax.set(xlabel='Average Movie Rating (1-10)', ylabel='Movie Gross Revenue (In Hundred Millions Of U.S. Dollars)')
sns.set(style='darkgrid', font_scale=2)
plt.show()

This bar chart checks if better-rated movies earn more. There's a loose trend, but it's not strictly linear — mid-rated films can still perform well.

### Popularity vs. Worldwide Gross

In [ ]:
#scatterplot of popularity vs. worldwide_gross
fig, ax = plt.subplots()
fig.set_size_inches(18,12)

ax = sns.scatterplot(x="popularity", y="worldwide_gross", marker="o", ci=68, data=merged2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(10))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set(xlabel='Movie Popularity (1-100)', ylabel='Movie Gross Revenue (U.S. Dollars)')
sns.set(style='darkgrid', font_scale=1.5)
plt.show()

This scatter plot examines popularity scores vs. revenue. Higher popularity often links with higher gross, but not always consistently.

###  Average Rating vs. Vote Count

In [ ]:
#scatterplot of vote_count vs. vote_average
fig, ax = plt.subplots()
fig.set_size_inches(8,12)

ax = sns.scatterplot(x="vote_average", y="vote_count", marker="o", ci=68, data=merged2)
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
ax.yaxis.set_major_locator(ticker.MultipleLocator(1000))
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.set(xlabel='Average Movie Rating (1-10)', ylabel='Number Of Votes')
sns.set(style='whitegrid', font_scale=1.2)
plt.show()

This scatter plot assesses whether higher-rated movies get more votes. Results suggest that while ratings influence interest, it's not the only driver of vote volume.

In [ ]:
votes_vs_gross = merged2[['vote_count', 'vote_average', 'worldwide_gross']]
votes_vs_gross_1000 = votes_vs_gross.loc[votes_vs_gross['vote_count'] >= 1000]
votes_vs_gross_1000 = votes_vs_gross_1000.round(0)
votes_vs_gross_1000

In [ ]:
votes_vs_gross_1000_grouped = votes_vs_gross_1000.groupby('vote_average', as_index=False).mean()
votes_vs_gross_1000_grouped

### Rating vs. Gross (When Votes > 1000)

In [ ]:
#scatterplot of vote_average, when vote_count is > 1000, vs. worldwide_gross
fig, ax = plt.subplots()
fig.set_size_inches(12,12)

ax = sns.barplot(x="vote_average", y="worldwide_gross", ci=None, data=votes_vs_gross_1000_grouped)
sns.set(style='whitegrid', font_scale=1.2)
sns.set_palette("Reds")
plt.yticks(ax.get_yticks(), ax.get_yticks()/1000000)
plt.ylabel('Average Movie Gross Revenue (In Millions Of U.S. Dollars)', fontsize=20)
plt.xlabel('Average Movie Rating (1-10)', fontsize=20)
plt.show()

Here, only well-voted movies are considered. Higher average ratings now show stronger correlation with revenue quality and attention both matter.

In [ ]:
genres_avg = merged2[['genres', 'worldwide_gross']]
genres_avg_grouped = genres_avg.groupby('genres', as_index=False).mean()
genres_avg_grouped_sorted = genres_avg_grouped.sort_values(by='worldwide_gross', ascending=True)
genres_avg_grouped_sorted

### Genre vs. Average Gross Revenue

In [ ]:
#scatterplot of genres vs. worldwide_gross (average)
sns.set(font_scale = 5)
sns.set(style="whitegrid", color_codes=True)
pal = sns.color_palette("Greens_d", len(genres_avg_grouped_sorted))
fig, ax = plt.subplots()
fig.set_size_inches(30,15)
ax = sns.barplot(x="genres", y="worldwide_gross", ci=None, palette=np.array(pal[::-1]),data=genres_avg_grouped_sorted)
#ax.set(xlabel='Movie Genre', ylabel='Average Movie Gross Revenue (U.S. Dollars)')
plt.xlabel('Movie Genres', fontsize=25, weight = 'bold')
plt.ylabel('Average Movie Gross Revenue (In Millions Of U.S. Dollars)', fontsize=25, weight = 'bold')
plt.yticks(ax.get_yticks(), ax.get_yticks()/1000000)
plt.tick_params(axis='both', which='major', labelsize=20)
#sns.set_style('darkgrid')
plt.show()

This bar chart shows which genres earn the most on average. Action and Adventure dominate, while niche genres tend to underperform financially.

In [ ]:
import ast
# 1. Create a sample DataFrame with genre_ids as strings
# This simulates the issue where the genre IDs are loaded as a string representation of a list.
data = df_tmdb_movies
tmdb_movies_df = pd.DataFrame(data)

# 2. Define a function to safely convert the string representation of a list to a real list
def parse_genre_ids(genre_ids_string):
    """
    Safely converts a string representation of a list to an actual list.
    Returns an empty list for any invalid input.
    """
    try:
        return ast.literal_eval(genre_ids_string)
    except (ValueError, SyntaxError):
        return []

# 3. Apply the parsing function to fix the data type
tmdb_movies_df['genre_ids'] = tmdb_movies_df['genre_ids'].apply(parse_genre_ids)

# The rest of the code is the same as the previous example to map the IDs to names.
# 4. Create a dictionary to map genre IDs to names
genre_id_to_name = {
    28: 'Action',
    12: 'Adventure',
    16: 'Animation',
    35: 'Comedy',
    80: 'Crime',
    99: 'Documentary',
    18: 'Drama',
    10751: 'Family',
    14: 'Fantasy',
    36: 'History',
    27: 'Horror',
    10402: 'Music',
    9648: 'Mystery',
    10749: 'Romance',
    878: 'Science Fiction',
    10770: 'TV Movie',
    53: 'Thriller',
    10752: 'War',
    37: 'Western'
}

# 5. Define a function to map the IDs to names
def map_genre_ids_to_names(genre_ids_list):
    """
    Maps a list of TMDB genre IDs to their corresponding names.
    Handles multiple genres per movie.
    """
    if isinstance(genre_ids_list, list):
        return [genre_id_to_name.get(gid, 'Unknown') for gid in genre_ids_list]
    return []

# 6. Apply the mapping function to the fixed 'genre_ids' column
tmdb_movies_df['genres'] = tmdb_movies_df['genre_ids'].apply(map_genre_ids_to_names)

# 7. Print the resulting DataFrame
tmdb_movies_df.genres

### Profit by Production Budget

In [ ]:
sns.set_theme(context='notebook', palette='blend:#7AB,#7AB', 
              style='white', font='sans-serif', font_scale=1.25, 
              color_codes=True, rc={'figure.figsize':(20,12)})

plot = sns.regplot(x='production_budget', y='profit', data=merged)
plt.axvline(0, color='black')
plt.axhline(0, color='black')

plot.set_title('Global Profit by Production Budget', fontsize=25)
plot.set_xlabel('Production Budget (MM)')
plot.set_ylabel('Global Profit (MM)');

plt.show()

This regression plot highlights how profit grows with increasing budget. While higher budgets can yield bigger profits, risk also rises.

In [ ]:
merged_ovmedian = movie_with_ratings.loc[
    movie_with_ratings['numvotes'] > movie_with_ratings['numvotes'].median()]

In [ ]:
merged_best_rated = merged_ovmedian.loc[(merged_ovmedian['averagerating'] >= 8.0
) 
                                                   & (merged_ovmedian['runtime_minutes'] < 250)]

In [ ]:
merged_best_rated

### Runtime of High-Rated Movies

In [ ]:
sns.set_theme(context='notebook', palette='blend:#7AB,#7AB', 
              style='white', font='sans-serif', font_scale=1.25, 
              color_codes=True, rc={'figure.figsize':(20,12)})

hist = sns.histplot(merged_best_rated['runtime_minutes'], bins=20)

hist.set(xlabel='Movie Length (min)', ylabel='Movie Count');
hist.set_title('Length of Movies Rated Higher than 8.0', fontsize=25);

This histogram examines the length of movies rated 8.0+. Most fall in the 90–130 minute range, suggesting a preferred window for impactful storytelling.

### MULTIVARIATE ANALYSIS

### Heatmap: Profit by Genre and Rating Category  

In [ ]:
merged2.columns

In [ ]:
merged2['rating_category'] = pd.cut(merged2['averagerating'],
                                         bins=[0, 6, 7.5, 10],
                                         labels=['Below Average', 'Good', 'Excellent'],
                                         right=False)

# Explode the genres column to have one row per genre
df_exploded = merged2.explode('genres')

# Group by both genre and rating category and find the average profit
pivot_table = df_exploded.pivot_table(
    values='profit',
    index='genres',
    columns='rating_category',
    aggfunc='mean'
).sort_values(by='Excellent', ascending=False)

print(pivot_table)

# Visualize the data using a heatmap for a clear comparison
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_table.dropna(), annot=True, fmt=".2f", cmap="YlGnBu")
plt.title('Average Profit by Genre and IMDb Rating Category')
plt.xlabel('IMDb Rating Category')
plt.ylabel('Genre')
plt.show()

This multivariate view shows how genre interacts with perceived quality (IMDb rating). Genres like Animation and Adventure stand out in the 'Excellent' category with higher average profit.

In [ ]:
df_exploded.to_csv('exploded_genres')

### Runtime vs Profit (Top Genres)  

In [ ]:
top_genres = df_exploded.groupby('genres')['profit'].mean().nlargest(5).index
df_top_genres = df_exploded[df_exploded['genres'].isin(top_genres)]

# Create a scatter plot of runtime vs. profit, colored by genre
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df_top_genres, x='runtime_minutes', y='profit', hue='genres', alpha=0.6)
plt.title('Profit vs. Runtime for Top 5 Genres')
plt.xlabel('Runtime (minutes)')
plt.ylabel('Profit (USD)')
plt.show()

For top genres, runtime correlates with profitability — especially in genres like Action or Adventure. Longer runtimes may justify higher ticket prices or appeal to core audiences.

### Gross vs Budget by Release Year  
This multivariate scatterplot shows production spending and gross revenue, segmented by release year. Temporal trends (like inflation or market saturation) may influence profitability across eras.

In [ ]:
# Create a regression plot to see the relationship over time
sns.relplot(
    data=merged2,
    x='production_budget',
    y='worldwide_gross',
    hue='release_year_x',
    palette='viridis',
    kind='scatter',
    height=6,
    aspect=1.5
)
plt.title('Worldwide Gross vs. Production Budget by Release Year')
plt.xlabel('Production Budget')
plt.ylabel('Worldwide Gross')
plt.show()